# 01 — Load PLAsTiCC Data

Download and verify the PLAsTiCC (Photometric LSST Astronomical Time-Series Classification Challenge) dataset.
Save to parquet for fast reloading in subsequent notebooks.

**Data source:** [PLAsTiCC Kaggle Competition](https://www.kaggle.com/competitions/PLAsTiCC-2018/data)
- Training set: ~7,800 labeled objects, 14 astrophysical classes
- 6 LSST passbands (ugrizy), irregularly sampled light curves
- Each observation: MJD, passband, flux, flux_err, detected flag

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to path
sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG
from src.data import load_training_metadata, load_training_lightcurves, get_object_lightcurve

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

## 1. Verify Data Files

Before running this notebook, download the PLAsTiCC data from Kaggle:
1. Go to https://www.kaggle.com/competitions/PLAsTiCC-2018/data
2. Download `training_set.csv` and `training_set_metadata.csv`
3. Place in `data/raw/`

In [ ]:
# Check that data files exist
for name, path in [("Training metadata", DATA_CONFIG["training_set_metadata"]),
                    ("Training light curves", DATA_CONFIG["training_set"])]:
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1e6 if exists else 0
    print(f"{name:25s}  {'FOUND' if exists else 'MISSING':8s}  {size:.1f} MB  ({path})")

## 2. Load Training Metadata

In [ ]:
metadata = load_training_metadata()
print(f"Metadata shape: {metadata.shape}")
print(f"\nColumns: {list(metadata.columns)}")
print(f"\nDtypes:\n{metadata.dtypes}")
metadata.head()

In [ ]:
# Null counts
print("Null counts:")
print(metadata.isnull().sum())
print(f"\nUnique objects: {metadata['object_id'].nunique()}")
print(f"Unique classes: {metadata['target'].nunique()}")
print(f"\nClass distribution:")
for target, name in sorted(DATA_CONFIG['class_names'].items()):
    count = (metadata['target'] == target).sum()
    print(f"  {target:3d} ({name:15s}): {count:5d}  ({100*count/len(metadata):.1f}%)")

## 3. Load Training Light Curves

In [ ]:
lc = load_training_lightcurves()
print(f"Light curve shape: {lc.shape}")
print(f"Columns: {list(lc.columns)}")
print(f"\nTotal observations: {len(lc):,}")
print(f"Unique objects: {lc['object_id'].nunique()}")
print(f"\nObservations per band:")
for band_id, band_name in DATA_CONFIG['bands'].items():
    count = (lc['passband'] == band_id).sum()
    print(f"  {band_name}: {count:,}")
lc.head()

In [ ]:
# Basic stats
print("Flux statistics:")
print(lc['flux'].describe())
print(f"\nFlux_err statistics:")
print(lc['flux_err'].describe())
print(f"\nMJD range: {lc['mjd'].min():.1f} — {lc['mjd'].max():.1f}")

## 4. Verify Join Integrity

In [ ]:
# Every object in LC should have metadata
lc_objects = set(lc['object_id'].unique())
meta_objects = set(metadata['object_id'].unique())
print(f"Objects in light curves: {len(lc_objects)}")
print(f"Objects in metadata: {len(meta_objects)}")
print(f"In LC but not metadata: {len(lc_objects - meta_objects)}")
print(f"In metadata but not LC: {len(meta_objects - lc_objects)}")

In [ ]:
# Observations per object
obs_per_object = lc.groupby('object_id').size()
print(f"Observations per object:")
print(obs_per_object.describe())

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(obs_per_object, bins=100, edgecolor='black', linewidth=0.3)
ax.set_xlabel('Number of Observations')
ax.set_ylabel('Number of Objects')
ax.set_title('Distribution of Observations per Object')
plt.tight_layout()
plt.show()

## 5. Quick Look: Example Light Curve

In [ ]:
from src.utils import plot_lightcurve, get_class_name

# Pick one example from each of a few classes
example_classes = [90, 92, 42, 88]  # SN Ia, RR Lyrae, SN II, AGN
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, target in zip(axes.flat, example_classes):
    obj_id = metadata[metadata['target'] == target]['object_id'].iloc[0]
    object_lc = get_object_lightcurve(obj_id, lc)
    class_name = get_class_name(target)
    plot_lightcurve(object_lc, object_id=obj_id,
                    title=f"{obj_id} — {class_name} (class {target})", ax=ax)

plt.tight_layout()
plt.show()

## 6. Save to Parquet

In [ ]:
os.makedirs(DATA_CONFIG['processed_dir'], exist_ok=True)

meta_path = os.path.join(DATA_CONFIG['processed_dir'], 'training_metadata.parquet')
lc_path = os.path.join(DATA_CONFIG['processed_dir'], 'training_lc.parquet')

metadata.to_parquet(meta_path, index=False)
lc.to_parquet(lc_path, index=False)

print(f"Saved metadata: {meta_path} ({os.path.getsize(meta_path)/1e6:.1f} MB)")
print(f"Saved light curves: {lc_path} ({os.path.getsize(lc_path)/1e6:.1f} MB)")